# Pertemuan 15 — Aproksimasi, Randomized, dan Algoritma Skala Besar
Aproksimasi Vertex Cover, Las Vegas vs Monte Carlo (Freivalds), **[Modern]** LSH/ANN dan kompleksitas self-attention.

In [ ]:
import itertools, math, random, time
import numpy as np
import matplotlib.pyplot as plt
random.seed(15); np.random.seed(15)

## 1. Vertex Cover: 2-aproksimasi (maximal matching) vs optimal
Ambil sembarang sisi (u,v) yang belum tercakup, masukkan **kedua** ujungnya. Jaminan: |VC| ≤ 2·OPT.

In [ ]:
def vc_approx(n, edges):
    cover, rem = set(), list(edges)
    for u, v in rem:
        if u not in cover and v not in cover: cover |= {u, v}
    return cover
def vc_opt(n, edges):
    for k in range(n + 1):
        for sub in itertools.combinations(range(n), k):
            s = set(sub)
            if all(u in s or v in s for u, v in edges): return k
ratios = []
for _ in range(60):
    n = 12; edges = [(u, v) for u, v in itertools.combinations(range(n), 2) if random.random() < 0.25]
    if edges: ratios.append(len(vc_approx(n, edges)) / vc_opt(n, edges))
print(f"rasio aproksimasi: rata-rata={np.mean(ratios):.3f}, terburuk={max(ratios):.3f}  (batas teori ≤ 2)")

## 2. Las Vegas vs Monte Carlo
- **Las Vegas** (Randomized Quick Sort): hasil selalu benar, waktu acak.
- **Monte Carlo** (Freivalds): waktu tetap O(n²) per putaran, hasil bisa salah dengan peluang ≤ 2⁻ᵏ.

In [ ]:
def qs_cost(a):
    if len(a) <= 1: return 0
    p = random.choice(a); L = [x for x in a if x < p]; G = [x for x in a if x > p]
    return len(a) + qs_cost(L) + qs_cost(G)
data = random.sample(range(10**6), 2000); costs = [qs_cost(data) for _ in range(300)]
print(f"Las Vegas: biaya rata-rata={np.mean(costs):.0f} (≈1.39·n·log2 n = {1.39*2000*math.log2(2000):.0f}), min={min(costs)}, maks={max(costs)}")

def freivalds(A, B, C, k=10):
    n = A.shape[0]
    for _ in range(k):
        r = np.random.randint(0, 2, n)
        if not np.array_equal(A @ (B @ r), C @ r): return False
    return True
n = 300; A, B = np.random.randint(0, 5, (n, n)), np.random.randint(0, 5, (n, n)); C = A @ B
Cwrong = C.copy(); Cwrong[5, 7] += 1
print("C benar →", freivalds(A, B, C), "| C salah (1 elemen) → terdeteksi?", not freivalds(A, B, Cwrong))
miss = sum(freivalds(A, B, Cwrong, k=1) for _ in range(2000)) / 2000
print(f"peluang lolos dengan 1 putaran ≈ {miss:.3f}  (teori ≤ 0.5);  dengan k putaran ≤ 0.5^k")

## 3. [Modern] Pencarian tetangga terdekat: eksak vs ANN (Random Hyperplane LSH)
Eksak: O(N·d) per query. LSH: hash vektor ke b bit (tanda proyeksi acak); hanya bandingkan kandidat dengan hash sama pada L tabel.

In [ ]:
N, d, nq = 30000, 64, 200
X = np.random.randn(N, d).astype(np.float32)
Q = X[np.random.choice(N, nq, replace=False)] + 0.3 * np.random.randn(nq, d).astype(np.float32)   # query ≈ titik data + derau
def exact_nn(q): return int(np.argmin(((X - q) ** 2).sum(1)))

class LSH:
    def __init__(self, L=10, b=12):
        self.planes = [np.random.randn(d, b).astype(np.float32) for _ in range(L)]
        self.tables = []
        for P in self.planes:
            codes = (X @ P > 0) @ (1 << np.arange(b)); T = {}
            for i, c in enumerate(codes): T.setdefault(int(c), []).append(i)
            self.tables.append(T)
    def query(self, q):
        cand = set()
        for P, T in zip(self.planes, self.tables): cand.update(T.get(int(((q @ P) > 0) @ (1 << np.arange(P.shape[1]))), []))
        if not cand: return -1, 0
        idx = np.fromiter(cand, int); return int(idx[np.argmin(((X[idx] - q) ** 2).sum(1))]), len(idx)

t = time.perf_counter(); truth = [exact_nn(q) for q in Q]; te = time.perf_counter() - t
for L, b in [(4, 14), (10, 12), (20, 10)]:
    lsh = LSH(L, b); t = time.perf_counter(); out = [lsh.query(q) for q in Q]; ta = time.perf_counter() - t
    recall = np.mean([o[0] == tr for o, tr in zip(out, truth)]); avg_c = np.mean([o[1] for o in out])
    print(f"LSH L={L:>2}, b={b:>2}: recall@1={recall:.2f} | kandidat rata-rata={avg_c:.0f}/{N} | waktu {ta:.2f}s vs eksak {te:.2f}s")

Opsional (butuh internet): FAISS dengan indeks HNSW.

In [ ]:
!pip -q install faiss-cpu

In [ ]:
import faiss
flat = faiss.IndexFlatL2(d); flat.add(X)
hnsw = faiss.IndexHNSWFlat(d, 32); hnsw.hnsw.efConstruction = 100; hnsw.add(X); hnsw.hnsw.efSearch = 64
_, I_flat = flat.search(Q, 1); _, I_h = hnsw.search(Q, 1)
print("recall@1 HNSW vs eksak:", float(np.mean(I_flat[:, 0] == I_h[:, 0])))
t = time.perf_counter(); flat.search(Q, 1); print("flat :", round(time.perf_counter() - t, 4), "s")
t = time.perf_counter(); hnsw.search(Q, 1); print("hnsw :", round(time.perf_counter() - t, 4), "s")

## 4. [Modern] Self-attention: O(n²) waktu & memori, varian efisien
Untuk panjang sekuens n dan dimensi d: standar = O(n²d) waktu dan O(n²) memori (matriks skor n×n). *Linear attention* = O(n d²). *Blockwise/online softmax* (ide FlashAttention) = hasil sama persis dengan memori O(n·blok).

In [ ]:
def attn_standard(Q, K, V):
    S = Q @ K.T / math.sqrt(Q.shape[1]); S -= S.max(1, keepdims=True); P = np.exp(S); P /= P.sum(1, keepdims=True); return P @ V
def attn_linear(Q, K, V):
    phi = lambda x: np.where(x > 0, x + 1, np.exp(x))               # elu(x)+1
    q, k = phi(Q), phi(K); kv = k.T @ V; z = q @ k.sum(0); return (q @ kv) / z[:, None]
def attn_blockwise(Q, K, V, bs=256):
    n, d = Q.shape; m = np.full(n, -np.inf); l = np.zeros(n); O = np.zeros((n, V.shape[1]))
    for s in range(0, n, bs):
        S = Q @ K[s:s+bs].T / math.sqrt(d); mn = np.maximum(m, S.max(1)); p = np.exp(S - mn[:, None]); sc = np.exp(m - mn)
        l = l * sc + p.sum(1); O = O * sc[:, None] + p @ V[s:s+bs]; m = mn
    return O / l[:, None]

d = 64; q, k, v = (np.random.randn(512, d).astype(np.float32) for _ in range(3))
print("blockwise == standar?", np.allclose(attn_standard(q, k, v), attn_blockwise(q, k, v), atol=1e-4))

ns = [256, 512, 1024, 2048, 4096]; ts, tl, mem = [], [], []
for n in ns:
    q, k, v = (np.random.randn(n, d).astype(np.float32) for _ in range(3))
    t = time.perf_counter(); attn_standard(q, k, v); ts.append(time.perf_counter() - t)
    t = time.perf_counter(); attn_linear(q, k, v);   tl.append(time.perf_counter() - t)
    mem.append(n * n * 4 / 2**20)
print("n     standar(s)  linear(s)  memori skor n×n (MiB, float32)")
for n, a, b, m_ in zip(ns, ts, tl, mem): print(f"{n:<5} {a:>10.4f} {b:>10.4f} {m_:>12.1f}")
print("kemiringan log-log waktu standar ≈ %.2f (→2), linear ≈ %.2f (→1)" % (np.polyfit(np.log(ns), np.log(ts), 1)[0], np.polyfit(np.log(ns), np.log(tl), 1)[0]))
plt.loglog(ns, ts, "o-", label="standar O(n²)"); plt.loglog(ns, tl, "s-", label="linear O(n)"); plt.legend(); plt.xlabel("panjang sekuens n"); plt.ylabel("detik"); plt.grid(True, which="both"); plt.show()

## 5. Latihan / Tugas Akhir
1. Ukur peningkatan *recall* LSH terhadap jumlah tabel L dan bit b; buat grafik recall vs waktu query.
2. Tunjukkan Karger's min-cut (Monte Carlo) pada graf kecil dan hitung peluang sukses empiris.
3. **Refleksi akhir:** buat peta konsep seluruh materi (P1–P15): untuk tiap teknik (brute force, D&C, greedy, DP, backtracking, B&B, aproksimasi/randomized) tuliskan satu komponen sistem AI yang memakainya, kompleksitasnya, dan batasannya.

In [ ]:
def karger_min_cut(n, edges):
    pass  # TODO